In [1]:
import librosa
import torch
import torchaudio

import matplotlib.pyplot as plt
import numpy as np
from torchvision import transforms
import torch.nn.functional as F

import pandas as pd
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import h5py

In [2]:
df = pd.read_feather('/kaggle/input/datasets/alexandra841/tracks-features/df_features_for_model', columns=['track_id', 'path', 'major_mood_class'])

In [3]:
paths = df['path'].tolist()

In [4]:
df['major_mood_class'].unique()

array(['calm', 'sad_dark', 'positive', 'energetic', 'deep_emotional',
       'romantic'], dtype=object)

In [5]:
mood_dict = dict(zip(df['major_mood_class'].unique(), range(len(df['major_mood_class'].unique()))))

In [6]:
mood_dict

{'calm': 0,
 'sad_dark': 1,
 'positive': 2,
 'energetic': 3,
 'deep_emotional': 4,
 'romantic': 5}

In [7]:
df['major_mood_class_int'] = df['major_mood_class'].map(mood_dict)

In [8]:
directory='/kaggle/input/datasets/alexandra841/mtg-tracks-data/data/'

In [9]:
class AudioDataSet(Dataset):

    def __init__(
        self, 
        paths, 
        df_meta, 
        directory='/kaggle/input/datasets/alexandra841/mtg-tracks-data/data/', 
        target_sr = 22050, 
        len_sec = 30
    ):

        self.paths = paths
        self.target_sample_rate = target_sr
        self.len_sec = len_sec
        self.directory=directory
        self.dict_track_info = df_meta.set_index("path").to_dict("index")
        self.resize = transforms.Compose([transforms.Resize((128, 256))])

        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=target_sr,
            n_fft=1024,
            hop_length=512,
            n_mels=128
        )

        self.db = torchaudio.transforms.AmplitudeToDB()

        
    def __len__(self):
        return len(self.paths)

    def preprocess(self, path):
        
        waveform, sr = torchaudio.load(path)
        
        max_len = self.target_sample_rate  * self.len_sec

        if sr != self.target_sample_rate:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=self.target_sample_rate)
            waveform = resampler(waveform)
            sr = self.target_sample_rate 

        len_audio = waveform.shape[1]
            
        if len_audio < max_len:
            pad = max_len - len_audio
            waveform = F.pad(waveform, (0, pad))
        else:
            waveform = waveform[:, :max_len]

        spec = self.db(self.mel(waveform))
        spec = self.resize(spec)
    
        return spec

    def get_track_info(self, path):
        short_path = path.replace(self.directory, '')
        track_info = self.dict_track_info[short_path]
        
        class_mood = track_info['major_mood_class']
        class_mood_int = track_info['major_mood_class_int']
        track_id = track_info['track_id']

        return class_mood, class_mood_int, track_id, short_path
        

    def __getitem__(self, idx):

        path = self.paths[idx]
        spec = self.preprocess(path)
        class_mood, class_mood_int, track_id, short_path = self.get_track_info(path)
        

        return {
            'path': short_path, 
            'track_id': track_id, 
            'spec': spec,
            'class_mood': class_mood,
            'class_mood_int': class_mood_int,
        }

In [10]:
dataset = AudioDataSet((directory + df['path']).tolist(), df)
loader = DataLoader(dataset, batch_size=32)

In [11]:
N = len(dataset)
str_dt = h5py.string_dtype(encoding="utf-8")

In [12]:
with h5py.File('/kaggle/working/features_resize.h5', 'w') as f:
    f.create_dataset('path', shape=(N,), dtype=str_dt)
    f.create_dataset('track_id', shape=(N,), dtype='int64')
    f.create_dataset('class_mood_int', shape=(N,), dtype="int8")
    f.create_dataset('class_mood', shape=(N,), dtype=str_dt)
    f.create_dataset(
        'spec',
        shape=(N,1,128,256),
        dtype="float16",
        compression="lzf"
    )

    idx=0

    for batch in tqdm(loader):

        batch_size = len(batch['track_id'])

        f['path'][idx:idx+batch_size] = batch['path']
        f['track_id'][idx:idx+batch_size] = batch['track_id'].numpy()
        f['class_mood_int'][idx:idx+batch_size] = batch['class_mood_int'].numpy()
        f['class_mood'][idx:idx+batch_size] = batch['class_mood']

        f['spec'][idx:idx+batch_size] = batch['spec'].numpy().astype(np.float16)

        idx += batch_size

100%|██████████| 488/488 [1:34:46<00:00, 11.65s/it]
